In [ ]:
import pandas as pd
import plotly.express as px

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/UPF_Deep_Learning_2026/final_project

Mounted at /content/drive
/content/drive/.shortcut-targets-by-id/13xDAtztvLxKEn8-JwTuL2NIfriCkru7U/UPF_Deep_Learning_2026/final_project


In [ ]:
df = pd.read_csv("data/HAM10000_metadata.csv")
display(df.head(10))

,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear
5,HAM_0001466,ISIC_0027850,bkl,histo,75.0,male,ear
6,HAM_0002761,ISIC_0029176,bkl,histo,60.0,male,face
7,HAM_0002761,ISIC_0029068,bkl,histo,60.0,male,face
8,HAM_0005132,ISIC_0025837,bkl,histo,70.0,female,back
9,HAM_0005132,ISIC_0025209,bkl,histo,70.0,female,back


In [ ]:

def classify_malignant_vs_benign(df):

    counts = df["dx"].value_counts().reset_index()
    counts.columns = ["dx", "count"]

    malignant = ['mel', 'bcc', 'akiec']
    counts["is_malignant"] = counts["dx"].isin(malignant)
    counts["type"] = counts["is_malignant"].map({True: "Malignant", False: "Benign"})

    labels_map = {
        'nv': 'Nevus', 'mel': 'Melanoma', 'bkl': 'Benign Keratosis',
        'bcc': 'BCC', 'akiec': 'AKIEC', 'vasc': 'Vascular', 'df': 'Dermatofibroma'
    }
    counts["dx_label"] = counts["dx"].map(labels_map)

    fig = px.bar(
        counts,
        x="dx_label",
        y="count",
        color="type",
        color_discrete_map={"Malignant": "red", "Benign": "blue"},
        title="Image distribution by skin lesion type",
        labels={"dx_label": "Diagnosis", "count": "No. of images", "type": ""},
        category_orders={"dx_label": list(labels_map.values())}
    )

    fig.write_html("figures/EDA_malignant_benign.html")


In [ ]:
def classify_malignant_vs_benign_by_diagnosis_method(df):

    malignant = ['mel', 'bcc', 'akiec']

    counts = df.groupby(["dx_type", "dx"]).size().reset_index(name="count")
    counts["is_malignant"] = counts["dx"].isin(malignant)
    counts["type"] = counts["is_malignant"].map({True: "Malignant", False: "Benign"})

    counts = counts.groupby(["dx_type", "type"])["count"].sum().reset_index()

    labels_map = {
        "histo"     : "Histology",
        "follow_up" : "Clinical follow-up",
        "consensus" : "Dermatologist consensus",
        "confocal"  : "Confocal microscopy"
    }
    counts["dx_type_label"] = counts["dx_type"].map(labels_map)

    fig = px.bar(
        counts,
        x="dx_type_label",
        y="count",
        color="type",
        color_discrete_map={"Malignant": "red", "Benign": "blue"},
        title="Image distribution by skin lesion type and diagnosis method",
        labels={"dx_type_label": "Diagnosis method", "count": "No. of images", "type": ""},
        category_orders={"dx_type_label": list(labels_map.values())}
    )

    fig.write_html("figures/EDA_malignant_benign_by_histology.html")




In [ ]:
def mean_age_by_dx(df):
    df_by_dx = df.groupby(["dx"]).agg(age_mean=("age", "mean"), age_std=("age", "std")).reset_index()
    df_by_dx["is_malignant"] = df_by_dx["dx"].isin(['mel', 'bcc', 'akiec'])
    fig = px.bar(
        df_by_dx,
        x="dx",
        y="age_mean",
        error_y="age_std",
        title="Mean age by skin lesion type",
        labels={"dx": "Diagnosis", "age_mean": "Mean age (years)"},
        color="is_malignant",
        color_discrete_map={True: "red", False: "blue"}
    )
    fig.write_html("figures/EDA_age_by_dx.html")





In [ ]:
def malignant_by_sex(df):
  df["malignant"] = False
  df.loc[df["dx"].isin(["mel", "bcc", "akiec"]), "malignant"] = True
  df_malignant = df[df["malignant"]]
  df_malignant_by_dx_sex = df_malignant.groupby(["dx", "sex"]).size().reset_index(name="count")
  fig = px.bar(
      df_malignant_by_dx_sex,
      x="dx",
      y="count",
      color="sex",
      color_discrete_map={"male": "green", "female": "orange"},
      title="Malignant lesion distribution by diagnosis type and sex",
      labels={"dx": "Diagnosis", "count": "No. of images"},
      barmode="group"
  )
  fig.write_html("figures/EDA_malignant_by_dx_sex.html")






In [ ]:
def top_malignant_locations(df):
    df["malignant"] = False
    list_malignant = ["mel", "bcc", "akiec"]
    df.loc[df["dx"].isin(list_malignant), "malignant"] = True
    colors = {"mel": "red", "bcc": "blue", "akiec": "yellow"}
    df_malignant = df[df["malignant"]]
    for c in list_malignant:
      df_malignant_localization = df_malignant[df_malignant["dx"] == c].groupby(["localization"]).size().reset_index(name="count")
      df_malignant_localization = df_malignant_localization.sort_values(by="count", ascending=False).head(6)
      fig = px.bar(
          df_malignant_localization,
          x="count",
          y="localization",
          orientation="h",
          color="localization",
          title=f"{c} - localization distribution",
          labels={"localization": "Body location", "count": "No. of images"}
      )
      fig.write_html(f"figures/EDA_malignant_{c}_location.html")



In [ ]:
def global_location_distribution(df):
  df_by_location_dx = df.groupby(["dx", "localization"]).size().reset_index(name="count")
  df_by_location_dx = df_by_location_dx.sort_values(by="count", ascending=False)
  fig = px.bar(
      df_by_location_dx,
      x="localization",
      y="count",
      color="dx",
      title="Global location distribution",
      labels={"dx": "Diagnosis", "count": "No. of images", "localization": "Body location"}
  )
  fig.write_html("figures/EDA_global_location.html")

In [ ]:
classify_malignant_vs_benign(df)
classify_malignant_vs_benign_by_diagnosis_method(df)
mean_age_by_dx(df)
malignant_by_sex(df)
top_malignant_locations(df)
#global_location_distribution(df)
